# Decision dataset (builder)

Builds typed-decision rows for `02_finetune_clef_flash.ipynb`. The model makes snap, step-level judgments from the goal and the last few turns, so every row is short and every label must be decidable from what the row shows.

Rules this builder enforces:

- **Nothing is truncated.** A row is counted with Clef's own tokenizer and prompt template (state, every question and every option). If it does not fit `token_limit`, it is skipped, never shrunk.
- **Labels are exact.** Each label is human-annotated, verified by execution, or made by an exact edit of a verified example, so the label is known by construction. No regex guesses and no LLM judges.
- **Splits never share an API or a source item.** Tool-call rows are split by the tool names they call, so the test split is unseen APIs.
- **No forced presentation.** Rows are rendered in several styles; one style is held out for testing only.

This build covers the tool-call families. Faithful-reporting rows (hook harness) and the step-error test sets (Who&When, TRAIL) are added in later builds.

In [ ]:
%pip install -q "transformers>=5.10.2" datasets pandas huggingface_hub wandb

In [ ]:
CFG = dict(
    # Which model the stored encodings are for. Rows, labels and splits are identical for every target; only `enc` differs.
    target_model="clef",               # "clef" (Cloudflare/clef-flash) or "d1" (LiquidAI/d1-3B)
    clef_repo="Cloudflare/clef-flash",
    d1_repo="LiquidAI/d1-3B",
    wandb_project="grounded-decision",
    log_to_wandb=True,
    seed=42,
    token_limit=16384,                 # Clef's context; a longer row is skipped, never shrunk
    xlam_rows=None,                    # None uses every xlam row; set an int for a quick build
    train_styles=("native_json", "transcript"),
    heldout_style="markdown",
    # Share of APIs / items per split (train, calib, val, test). Eval splits are then sampled down per stratum,
    # so they only need enough items to fill the caps; When2Call is small and keeps a larger share.
    split_shares={"xlam": (90, 3, 3, 4), "when2call": (80, 5, 5, 10)},
    # Rows kept per (family, kind) stratum in each eval split, so rare kinds get as many rows as common ones.
    eval_caps={"val": 60, "calib": 150, "test": 150},
    # Agent traces, BFCL and grounding rows, built and audited by builders/ (one module per source) and logged to W&B.
    sources_artifact="decision-sources:v2",   # v2: BFCL multi-turn ask/decline split by scenario, with train rows
    sources_dir=None,                  # a local folder written by builders/ (outputs/sources); None: download sources_artifact
    # A run sees a few thousand training rows, so the mix matters more than the totals: xlam train rows are capped, and
    # the smaller new families are shown more than once (copies keep their labels; eval rows are never repeated). A
    # repeat key is a source, or "source:stratum prefix" for part of one (the most specific match wins). When2Call and
    # BFCL multi-turn are repeated because v2 (shown each once, multi-turn never) lost call/ask accuracy to "decline".
    train_caps={"xlam": 8000},
    train_repeat={"agentdojo": 2, "tau_bench": 2, "halueval": 2, "when2call": 2, "bfcl:multi_turn": 3},
)
CFG["out_dir"] = {"clef": "outputs/decision-dataset-v1", "d1": "outputs/decision-dataset-d1"}[CFG["target_model"]]
CFG["artifact_name"] = {"clef": "decision-dataset", "d1": "decision-dataset-d1"}[CFG["target_model"]]
CHECKS_PASSED = False  # set by the hard checks; nothing is written or logged until they pass

In [ ]:
import hashlib, json, random, re, sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

# Tokenizer, prompt template and encoder come from the Clef repo itself; weights are not downloaded.
CLEF = Path(snapshot_download(CFG["clef_repo"], allow_patterns=["*.py", "*.json", "*.jinja"]))
sys.path.insert(0, str(CLEF))
import joint_schema_model as jsm
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(CLEF)
OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
REVISIONS = {}


def encode_clef(state, questions, gold):
    """Clef's exact encoding: system text + state + every question and option, read jointly in one sequence."""
    enc = jsm.encode_record(tok, {"id": "x", "state": state, "questions": questions}, max_length=10**9)
    return len(enc.input_ids), {
        "length": len(enc.input_ids), "input_ids": list(enc.input_ids),
        "questions": [{"id": q.question_id, "type": q.question_type, "span": list(q.question_span),
                       "option_spans": [list(sp) for sp in q.option_spans], "option_ids": list(q.option_ids),
                       "label": q.option_ids.index(gold_key(questions[q.question_id], gold[q.question_id]))}
                      for q in enc.questions]}


if CFG["target_model"] == "d1":
    import importlib.util
    # d1's own prompt code renders the training prompts, so they match what d1 reads at inference exactly.
    D1 = Path(snapshot_download(CFG["d1_repo"], allow_patterns=["*.py", "*.json", "*.jinja"]))
    spec = importlib.util.spec_from_file_location("d1_prompt", D1 / "prompt.py")
    d1_prompt = importlib.util.module_from_spec(spec)
    sys.modules["d1_prompt"] = d1_prompt  # its dataclasses look their module up while it loads
    spec.loader.exec_module(d1_prompt)
    tok_d1 = AutoTokenizer.from_pretrained(D1)
    D1_BOS = tok_d1.bos_token if isinstance(tok_d1.bos_token, str) else ""


def d1_label(question, gold):
    """Gold option index in d1's readout order: yes before no, choices in criteria order, score levels by index."""
    if question["type"] == "noul":
        return 0 if gold else 1
    if question["type"] == "choice":
        return list(question["criteria"]).index(str(gold))
    return int(gold)


def encode_d1(state, questions, gold):
    """d1 asks one question per sequence: state, question, options as codes, then the answer slot. The answer is the
    softmax over each option's best token logit there (`groups`)."""
    out = []
    for qid, question in questions.items():
        q = d1_prompt.as_question(question)
        text = d1_prompt.render(tok_d1, state, q, D1_BOS, d1_prompt.default_lead("lfm2_vl"), d1_prompt.DEFAULT_STATE_STYLE,
                                d1_prompt.DEFAULT_SYSTEM, "desc")
        groups = [list(g) for g in d1_prompt.readout_ids(tok_d1, q)]
        label = d1_label(question, gold[qid])
        assert 0 <= label < len(groups), (qid, label, len(groups))
        out.append({"id": qid, "input_ids": tok_d1.encode(text, add_special_tokens=False), "groups": groups, "label": label})
    length = max(len(q["input_ids"]) for q in out)
    return length, {"length": length, "questions": out}


encode_row = {"clef": encode_clef, "d1": encode_d1}[CFG["target_model"]]


def gid(*parts):
    return hashlib.sha1("\x1f".join(map(str, parts)).encode()).hexdigest()[:16]


def split_of(key, source):
    train, calib, val, _ = CFG["split_shares"][source]
    b = int(gid(key), 16) % 100
    return "train" if b < train else "calib" if b < train + calib else "val" if b < train + calib + val else "test"


def gold_key(question, gold):
    return ("true" if gold else "false") if question["type"] == "noul" else str(gold)

## Tool-call correctness (xlam-function-calling-60k)

Every xlam answer was verified by execution. Each source item becomes one row: either the verified calls, or the verified calls with exactly one edit that makes them wrong by construction.

| Kind | Edit | Why the label is certain |
|---|---|---|
| `correct` | none | verified by execution |
| `wrong_tool` | one call's tool swapped for another listed tool whose parameters do not include the call's arguments | the call does not match the swapped tool's schema |
| `wrong_arguments` / undefined parameter | a parameter the tool does not define is added | the call does not match the tool's schema |
| `wrong_arguments` / missing value | a required argument whose value the user typed is removed | the call ignores a value the user gave |
| `wrong_arguments` / changed value | an argument whose value the user typed is replaced by one the user did not type | the call contradicts the user |
| `incomplete` | one of several distinct calls is dropped | part of the request is not done |

The split is decided by tool name. A row is kept only if every tool it calls falls in the same split, so the test split contains only APIs never seen in training.

In [ ]:
repo = "Salesforce/xlam-function-calling-60k"
REVISIONS[repo] = HfApi().dataset_info(repo).sha
XLAM = json.load(open(hf_hub_download(repo, "xlam_function_calling_60k.json", repo_type="dataset", revision=REVISIONS[repo])))
for r in XLAM:
    r["answers"], r["tools"] = json.loads(r["answers"]), json.loads(r["tools"])

# Exact duplicate requests would put the same item in several rows; keep the first.
seen, items = set(), []
for r in XLAM:
    key = r["query"].strip().lower()
    if key not in seen:
        seen.add(key); items.append(r)
print(len(XLAM), "xlam rows,", len(XLAM) - len(items), "duplicate requests removed")

# Values seen per parameter name, used to replace a value with a realistic one the user did not type.
VALUE_POOL = defaultdict(set)
for r in items:
    for call in r["answers"]:
        for name, value in call["arguments"].items():
            if isinstance(value, (str, int, float)) and not isinstance(value, bool):
                VALUE_POOL[name].add(value)
VALUE_POOL = {k: sorted(v, key=str) for k, v in VALUE_POOL.items()}
PARAM_NAMES = sorted({p for r in items for t in r["tools"] for p in t["parameters"]})


def typed_by_user(value, query):
    """True when the value appears literally in the request, so the user asked for exactly this value."""
    if isinstance(value, bool) or not isinstance(value, (str, int, float)):
        return False
    text = str(value).strip()
    if len(text) < 2 and not text.isdigit():
        return False
    return re.search(r"(?<![\w.])" + re.escape(text) + r"(?![\w])", query, re.I) is not None


def tool_by_name(tools):
    return {t["name"]: t for t in tools}


def required(param):
    return "optional" not in str(param.get("type", "")).lower()


def edit_wrong_tool(r, rng):
    tools = tool_by_name(r["tools"])
    options = []
    for i, call in enumerate(r["answers"]):
        for other in tools.values():
            if other["name"] != call["name"] and not set(call["arguments"]) <= set(other["parameters"]):
                options.append((i, other["name"]))
    if not options:
        return None
    i, name = rng.choice(options)
    calls = [dict(c) for c in r["answers"]]
    calls[i] = {"name": name, "arguments": calls[i]["arguments"]}
    return calls, "wrong_tool", "swapped_tool"


def edit_undefined_parameter(r, rng):
    tools = tool_by_name(r["tools"])
    i = rng.randrange(len(r["answers"]))
    call = r["answers"][i]
    schema = tools.get(call["name"], {}).get("parameters", {})
    candidates = [p for p in rng.sample(PARAM_NAMES, min(50, len(PARAM_NAMES)))
                  if p not in schema and p not in call["arguments"] and VALUE_POOL.get(p)]
    if not candidates:
        return None
    name = candidates[0]
    calls = [dict(c) for c in r["answers"]]
    calls[i] = {"name": call["name"], "arguments": {**call["arguments"], name: rng.choice(VALUE_POOL[name])}}
    return calls, "wrong_arguments", "undefined_parameter"


def edit_missing_value(r, rng):
    tools = tool_by_name(r["tools"])
    options = [(i, name) for i, call in enumerate(r["answers"]) for name, value in call["arguments"].items()
               if typed_by_user(value, r["query"])
               and required(tools.get(call["name"], {}).get("parameters", {}).get(name, {}))]
    if not options:
        return None
    i, name = rng.choice(options)
    calls = [dict(c) for c in r["answers"]]
    calls[i] = {"name": calls[i]["name"], "arguments": {k: v for k, v in calls[i]["arguments"].items() if k != name}}
    return calls, "wrong_arguments", "missing_value"


def edit_changed_value(r, rng):
    options = [(i, name, value) for i, call in enumerate(r["answers"]) for name, value in call["arguments"].items()
               if typed_by_user(value, r["query"])]
    rng.shuffle(options)
    for i, name, value in options:
        if isinstance(value, (int, float)):
            pool = [value + d for d in (1, 2, 3, 5, 10, -1) if value + d >= 0]
        else:
            pool = [v for v in VALUE_POOL.get(name, []) if isinstance(v, str)]
        pool = [v for v in pool if str(v).lower() != str(value).lower() and not typed_by_user(v, r["query"])]
        if pool:
            calls = [dict(c) for c in r["answers"]]
            calls[i] = {"name": calls[i]["name"], "arguments": {**calls[i]["arguments"], name: rng.choice(pool)}}
            return calls, "wrong_arguments", "changed_value"
    return None


def edit_incomplete(r, rng):
    distinct = {json.dumps(c, sort_keys=True) for c in r["answers"]}
    if len(r["answers"]) < 2 or len(distinct) < len(r["answers"]):
        return None
    i = rng.randrange(len(r["answers"]))
    return [c for j, c in enumerate(r["answers"]) if j != i], "incomplete", "dropped_call"


EDITS = [edit_wrong_tool, edit_undefined_parameter, edit_missing_value, edit_changed_value, edit_incomplete]

## Rendering and questions

The same row can be shown as JSON, as a transcript, or as markdown (held out). Questions come in several wordings and answer types, sampled per row.

In [ ]:
def render_tool_row(query, tools, calls, style):
    if style == "native_json":
        return {"tools": tools, "user": query, "assistant_tool_calls": calls}
    call_lines = "\n".join(f'-> {c["name"]}({json.dumps(c["arguments"], ensure_ascii=False)})' for c in calls)
    if style == "transcript":
        return ("TOOLS:\n" + "\n".join(json.dumps(t, ensure_ascii=False) for t in tools)
                + f"\n\nUSER: {query}\n\nASSISTANT (tool calls):\n{call_lines}")
    tool_lines = "\n".join(f'- `{t["name"]}`: {t["description"]} Parameters: {json.dumps(t["parameters"], ensure_ascii=False)}'
                           for t in tools)
    return f"## Available tools\n{tool_lines}\n\n## User request\n{query}\n\n## Proposed tool calls\n```\n{call_lines}\n```"


CALL_KINDS = {"correct": "the calls are right and do everything the user asked",
              "wrong_tool": "a call uses a tool that cannot do what was asked",
              "wrong_arguments": "the right tool is called with arguments that do not match the request or the tool",
              "incomplete": "the calls are right but part of the request is not done"}


def q_call_ok(rng, kind):
    wording = rng.choice(["Are the assistant's tool calls correct and complete for the user's request?",
                          "Should these tool calls be executed exactly as proposed?",
                          "Do the proposed calls do exactly what the user asked, with valid arguments?"])
    return {"type": "noul", "instructions": wording}, kind == "correct"


def q_call_kind(rng, kind):
    return ({"type": "choice", "instructions": rng.choice(["What, if anything, is wrong with the proposed tool calls?",
                                                           "Classify the assistant's tool calls."]),
             "criteria": CALL_KINDS}, kind)


def make_row(rid, source, group, split, state, questions, gold, meta, enc):
    return {"id": rid, "source": source, "group": group, "split": split, "state": state,
            "questions": questions, "gold": gold, "meta": meta, "enc": enc}

In [ ]:
rng = random.Random(CFG["seed"])
ROWS, REPORT = [], []
pool = items if CFG["xlam_rows"] is None else rng.sample(items, CFG["xlam_rows"])
for r in pool:
    splits = {split_of(("api", c["name"]), "xlam") for c in r["answers"]}
    if len(splits) != 1:
        REPORT.append({"source": "xlam", "dropped": "calls APIs from two splits"}); continue
    split = splits.pop()
    row_rng = random.Random(gid(CFG["seed"], "xlam", r["id"]))
    if row_rng.random() < 0.5:
        calls, kind, edit = r["answers"], "correct", None
    else:
        result = None
        for edit_fn in row_rng.sample(EDITS, len(EDITS)):
            result = edit_fn(r, row_rng)
            if result:
                break
        if result is None:
            REPORT.append({"source": "xlam", "dropped": "no exact edit applies"}); continue
        calls, kind, edit = result
    plans = [(split, row_rng.choice(CFG["train_styles"]))]
    if split == "test":
        plans.append(("test_heldout_format", CFG["heldout_style"]))
    for out_split, style in plans:
        questions, gold = {}, {}
        for name, make in (("call_ok", q_call_ok), ("call_kind", q_call_kind)):
            if name == "call_ok" or row_rng.random() < 0.5:
                questions[name], gold[name] = make(row_rng, kind)
        state = render_tool_row(r["query"], r["tools"], calls, style)
        tokens, enc = encode_row(state, questions, gold)
        if tokens > CFG["token_limit"]:
            REPORT.append({"source": "xlam", "split": out_split, "dropped": "over token limit", "tokens": tokens}); continue
        ROWS.append(make_row(f'xlam:{r["id"]}:{out_split}', "xlam", "|".join(sorted({c["name"] for c in r["answers"]})),
                             out_split, state, questions, gold,
                             {"tokens": tokens, "style": style, "kind": kind, "edit": edit, "calls": calls,
                              "families": {"call_ok": "tool_call_ok", "call_kind": "tool_call_kind"}}, enc))
        REPORT.append({"source": "xlam", "split": out_split, "kind": kind, "edit": edit, "tokens": tokens, "dropped": None})
print(sum(r["source"] == "xlam" for r in ROWS), "xlam rows")

## Call, ask or decline (When2Call)

Given the tools and a request, should the assistant call a tool, ask for missing information, decline, or answer directly? The reference answers are curated. This uses When2Call's multiple-choice test split, so it is no longer an untouched benchmark for us.

In [ ]:
repo = "nvidia/When2Call"
REVISIONS[repo] = HfApi().dataset_info(repo).sha
DESCRIPTIONS = {"tool_call": "call one of the available tools", "request_for_info": "ask the user for the missing information",
                "cannot_answer": "say this cannot be done with the tools available", "direct": "answer directly without a tool"}
ALT_KEYS = {"tool_call": "call", "request_for_info": "ask", "cannot_answer": "decline", "direct": "answer"}
for ex in load_dataset(repo, "test", split="mcq", revision=REVISIONS[repo]):
    keys = [k for k, v in ex["answers"].items() if v]
    assert ex["correct_answer"] in keys, ex["uuid"]
    row_rng = random.Random(gid(CFG["seed"], "when2call", ex["uuid"]))
    rename = ALT_KEYS if row_rng.random() < 0.5 else {k: k for k in DESCRIPTIONS}
    tools = [json.loads(t) for t in ex["tools"]]
    split = split_of(("when2call", ex["source_id"] or ex["uuid"]), "when2call")
    plans = [(split, row_rng.choice(CFG["train_styles"]))] + ([("test_heldout_format", CFG["heldout_style"])] if split == "test" else [])
    for out_split, style in plans:
        if style == "native_json":
            state = {"tools": tools, "user": ex["question"]}
        elif style == "transcript":
            state = "TOOLS:\n" + "\n".join(json.dumps(t, ensure_ascii=False) for t in tools) + "\n\nUSER: " + ex["question"]
        else:
            state = ("## Available tools\n" + "\n".join(f'- `{t["name"]}`: {t.get("description", "")}' for t in tools)
                     + "\n\n## User request\n" + ex["question"])
        questions = {"next_step": {"type": "choice",
                                   "instructions": row_rng.choice(["What should the assistant do next?",
                                                                   "Pick the right next move for the assistant.",
                                                                   "Given these tools and this request, how should the assistant respond?"]),
                                   "criteria": {rename[k]: DESCRIPTIONS[k] for k in keys}}}
        gold = {"next_step": rename[ex["correct_answer"]]}
        tokens, enc = encode_row(state, questions, gold)
        if tokens > CFG["token_limit"]:
            REPORT.append({"source": "when2call", "split": out_split, "dropped": "over token limit", "tokens": tokens}); continue
        ROWS.append(make_row(f'when2call:{ex["uuid"]}:{out_split}', "when2call", ex["source_id"] or ex["uuid"], out_split,
                             state, questions, gold,
                             {"tokens": tokens, "style": style, "kind": ex["correct_answer"], "edit": None,
                              "families": {"next_step": "call_ask_decline"}}, enc))
        REPORT.append({"source": "when2call", "split": out_split, "kind": ex["correct_answer"], "tokens": tokens, "dropped": None})
print(sum(r["source"] == "when2call" for r in ROWS), "When2Call rows")

## Agent traces, BFCL and grounding (decision-sources)

Built by `builders/` in the repo, one module per source, each with exact labels, task-level splits and its own checks, then logged to W&B as `decision-sources`. This cell downloads a pinned version and encodes its rows for the target model.

| Source | Question | What a row shows | Label |
|---|---|---|---|
| AgentDojo (MIT) | Is this call a correct next step? | goal, optionally the last two whole steps, the proposed call | the benchmark's ground truth: an injected action (attack confirmed by AgentDojo's checker) or a user-task call |
| tau-bench (MIT) | Is this call a correct next step? | every user message, the fewest last steps that make the label decidable, the proposed write | an expected action, a real wrong-target write, or a target swap of an expected action |
| BFCL v4 (Apache-2.0) | call_ok / call_kind; call or not; call, ask or decline (test) | tools, request, calls | the answer key, with xlam's exact edits; curated irrelevance; multi-turn ask/decline turns located from the answer keys |
| HaluEval (MIT) | Is the response supported by the context? | context, user, response | human-built pairs, length-matched, kept only when support is decidable from the context shown |

In [ ]:
import wandb

if CFG["sources_dir"]:
    SOURCES_DIR = Path(CFG["sources_dir"])
    manifest_file = SOURCES_DIR / "manifest.json"  # written by builders/log_sources.py, if it was run
    REVISIONS["decision-sources"] = {"version": "local",
                                     **(json.loads(manifest_file.read_text())["upstream"] if manifest_file.exists() else {})}
else:
    sources = wandb.Api().artifact(f'{CFG["wandb_project"]}/{CFG["sources_artifact"]}', type="dataset")
    SOURCES_DIR = Path(sources.download(root=str(OUT.parent / "decision-sources")))
    REVISIONS["decision-sources"] = {"version": sources.version, **sources.metadata["upstream"]}
n_before = len(ROWS)
for name in ("agentdojo", "tau_bench", "bfcl", "grounding"):
    for line in open(SOURCES_DIR / f"{name}.jsonl"):
        row = json.loads(line)
        tokens, enc = encode_row(row["state"], row["questions"], row["gold"])
        if tokens > CFG["token_limit"]:
            REPORT.append({"source": row["source"], "split": row["split"], "dropped": "over token limit", "tokens": tokens}); continue
        row["meta"]["tokens"] = tokens
        row["enc"] = enc
        ROWS.append(row)
        REPORT.append({"source": row["source"], "split": row["split"], "kind": row["meta"]["kind"], "edit": row["meta"]["edit"],
                       "tokens": tokens, "dropped": None})
print(len(ROWS) - n_before, "rows from", SOURCES_DIR, REVISIONS["decision-sources"]["version"])

## Stratified evaluation splits

Validation, calibration and test are sampled down to a fixed number of rows per stratum (source x kind of example: each edit type, correct calls, and each When2Call answer), so rare kinds are measured as precisely as common ones. Strata smaller than the cap keep every row. The held-out-format test rows are the same items as the sampled test rows.

In [ ]:
def stratum(row):
    return (row["source"], row["meta"].get("stratum") or row["meta"]["edit"] or row["meta"]["kind"])


def item(row):
    return row["id"].rsplit(":", 1)[0]


sample_rng = random.Random(CFG["seed"])
kept, dropped = [], Counter()
for split, cap in CFG["eval_caps"].items():
    by = defaultdict(list)
    for row in ROWS:
        if row["split"] == split:
            by[stratum(row)].append(row)
    for key, rows in sorted(by.items()):
        rows = sorted(rows, key=lambda r: r["id"])
        sample_rng.shuffle(rows)
        kept += rows[:cap]
        dropped[split] += max(0, len(rows) - cap)
test_items = {item(r) for r in kept if r["split"] == "test"}
kept += [r for r in ROWS if r["split"] == "test_heldout_format" and item(r) in test_items]
dropped["test_heldout_format"] = sum(r["split"] == "test_heldout_format" for r in ROWS) - len(test_items)
def repeat_of(row):
    """Times a train row is shown: the most specific train_repeat key ("source:stratum prefix", then "source")."""
    source, part = stratum(row)
    keys = [k for k in CFG["train_repeat"] if k == source or (k.startswith(f"{source}:") and part.startswith(k.split(":", 1)[1]))]
    return CFG["train_repeat"][max(keys, key=len)] if keys else 1


train = [r for r in ROWS if r["split"] == "train"]
mixed = []
for source in sorted({r["source"] for r in train}):
    rows = sorted([r for r in train if r["source"] == source], key=lambda r: r["id"])
    cap = CFG["train_caps"].get(source)
    if cap is not None and len(rows) > cap:
        random.Random(f'{CFG["seed"]}|cap|{source}').shuffle(rows)
        dropped["train (capped)"] += len(rows) - cap
        rows = rows[:cap]
    mixed += rows
    for r in rows:
        for k in range(2, repeat_of(r) + 1):  # copies, with their own ids
            mixed.append({**r, "id": f'{r["id"]}#copy{k}', "meta": {**r["meta"], "copy": k}})
ROWS = mixed + kept
print("Rows dropped by sampling:", dict(dropped))
print("Train rows by family:", Counter(f for r in ROWS if r["split"] == "train" for f in r["meta"]["families"].values()))
print(pd.DataFrame([{"split": r["split"], "stratum": ":".join(stratum(r))} for r in ROWS if r["split"] != "train"])
      .groupby(["stratum", "split"]).size().unstack(fill_value=0))

## Report and hard checks

The hard checks must pass before anything is written or logged.

In [ ]:
rep = pd.DataFrame(REPORT)
print("Kept and dropped by source:")
print(rep.assign(kept=rep["dropped"].isna()).groupby(["source", "kept"]).size().unstack(fill_value=0))
print("\nDrop reasons:")
print(rep["dropped"].value_counts())
print("\nToken length of kept rows (percentiles):")
print(rep[rep["dropped"].isna()].groupby("source")["tokens"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

kinds = pd.DataFrame([{"source": r["source"], "split": r["split"], "kind": r["meta"]["kind"], "edit": r["meta"]["edit"]} for r in ROWS])
print("\nLabels by split (look for splits that are nearly all one answer):")
print(kinds.groupby(["source", "split", "kind"]).size().unstack(fill_value=0))
print("\nEdits used for wrong xlam rows:")
print(kinds[kinds["source"] == "xlam"]["edit"].value_counts(dropna=False))

In [ ]:
ids = Counter(row["id"] for row in ROWS)
assert len(ids) == len(ROWS), f"duplicate row ids: {[i for i, n in ids.items() if n > 1][:5]}"

home = {}
for row in ROWS:
    for name, question in row["questions"].items():
        options = [key for key, _ in jsm.question_options(question)]
        assert gold_key(question, row["gold"][name]) in options, (row["id"], name, row["gold"][name], options)
    assert row["meta"]["tokens"] <= CFG["token_limit"], row["id"]
    base = "test" if row["split"].startswith("test") else row["split"]
    keys = [("api", n) for n in row["group"].split("|")] if row["source"] == "xlam" else [("item", row["source"], row["group"])]
    for key in keys:
        assert home.setdefault(key, base) == base, f"{key} is in two splits"
    if row["split"] != "test_heldout_format":
        assert row["meta"]["style"] != CFG["heldout_style"], row["id"]

# Re-encode a sample: the stored encoding and labels must match a fresh encode exactly.
for row in random.Random(0).sample(ROWS, min(1000, len(ROWS))):
    assert encode_row(row["state"], row["questions"], row["gold"]) == (row["meta"]["tokens"], row["enc"]), row["id"]
    if CFG["target_model"] == "clef":  # and Clef's own encoder at the real limit cuts nothing
        enc = jsm.encode_record(tok, {"id": row["id"], "state": row["state"], "questions": row["questions"]}, max_length=CFG["token_limit"])
        assert list(enc.input_ids) == row["enc"]["input_ids"], row["id"]
for row in ROWS:
    assert row["enc"]["length"] == row["meta"]["tokens"] <= CFG["token_limit"], row["id"]

# Re-check every xlam row against its own construction rule, independently of the edit code.
by_id = {r["id"]: r for r in items}
for row in ROWS:
    if row["source"] != "xlam":
        continue
    src, calls, edit = by_id[int(row["id"].split(":")[1])], row["meta"]["calls"], row["meta"]["edit"]
    tools = tool_by_name(src["tools"])
    assert row["gold"]["call_ok"] is (row["meta"]["kind"] == "correct"), row["id"]
    if edit is None:
        assert calls == src["answers"], row["id"]
        continue
    changed = [(a, b) for a, b in zip(src["answers"], calls) if a != b]
    if edit == "dropped_call":
        assert len(calls) == len(src["answers"]) - 1 and all(c in src["answers"] for c in calls), row["id"]
        continue
    assert len(calls) == len(src["answers"]) and len(changed) == 1, row["id"]
    a, b = changed[0]
    if edit == "swapped_tool":
        assert b["name"] != a["name"] and b["name"] in tools and not set(b["arguments"]) <= set(tools[b["name"]]["parameters"]), row["id"]
    elif edit == "undefined_parameter":
        assert b["name"] == a["name"] and set(b["arguments"]) - set(tools[a["name"]]["parameters"]), row["id"]
    elif edit == "missing_value":
        (gone,) = set(a["arguments"]) - set(b["arguments"])
        assert typed_by_user(a["arguments"][gone], src["query"]), row["id"]
    elif edit == "changed_value":
        (key,) = [k for k in a["arguments"] if a["arguments"][k] != b["arguments"].get(k)]
        assert typed_by_user(a["arguments"][key], src["query"]) and not typed_by_user(b["arguments"][key], src["query"]), row["id"]

# decision-sources rows: each label must agree with the kind its builder assigned.
KIND_GOLD = {"step_ok": lambda k: k == "correct_step", "supported": lambda k: k == "supported", "call_ok": lambda k: k == "correct"}
for row in ROWS:
    if row["source"] in ("xlam", "when2call"):
        continue
    for name, check in KIND_GOLD.items():
        if name in row["gold"]:
            assert row["gold"][name] is check(row["meta"]["kind"]), row["id"]
    assert not row["split"].startswith("test") or not row["meta"].get("copy"), row["id"]

for split in ("train", "calib", "val", "test"):
    assert any(r["split"] == split for r in ROWS), f"split {split} is empty"
CHECKS_PASSED = True
print("Hard checks passed.")

In [ ]:
def show(row):
    state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"], ensure_ascii=False, indent=1)
    print("=" * 100)
    print(row["id"], "|", row["meta"]["style"], "|", row["meta"]["tokens"], "tokens |", row["meta"]["kind"], row["meta"]["edit"] or "")
    print(state[-1500:])
    for name, question in row["questions"].items():
        print(f'\nQ [{row["meta"]["families"][name]}] {question["instructions"]}\n  GOLD: {row["gold"][name]}')


# Read these: two rows of every kind and edit.
groups = defaultdict(list)
for row in ROWS:
    groups[(row["source"], row["meta"]["kind"], row["meta"]["edit"])].append(row)
for key in sorted(groups, key=str):
    for row in random.Random(1).sample(groups[key], min(2, len(groups[key]))):
        show(row)

In [ ]:
# Notebook runners may keep going after a failed cell; never write or log a dataset that failed its checks.
assert CHECKS_PASSED, "the hard checks did not pass; fix them before writing or logging the dataset"
SPLITS = defaultdict(list)
for row in ROWS:
    SPLITS[row["split"]].append(row)
for name, rows in SPLITS.items():
    random.Random(CFG["seed"]).shuffle(rows)
    with open(OUT / f"{name}.jsonl", "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
manifest = {
    "config": CFG,
    "clef_revision": CLEF.name,
    "target_model": CFG["target_model"],
    "model_repo": {"clef": CFG["clef_repo"], "d1": CFG["d1_repo"]}[CFG["target_model"]],
    "model_revision": CLEF.name if CFG["target_model"] == "clef" else D1.name,
    "source_revisions": REVISIONS,
    "rows": {name: len(rows) for name, rows in SPLITS.items()},
    "questions_by_family": Counter(f for row in ROWS for f in row["meta"]["families"].values()),
    "split_rule": "sha1(api name or source item) % 100 into train/calib/val/test by split_shares per source; decision-sources rows keep their builders' task-level splits (AgentDojo: banking suite held out)",
    "train_mix": {"caps": CFG["train_caps"], "repeat": CFG["train_repeat"]},
    "eval_sampling": "val/calib/test capped at eval_caps rows per (source, kind) stratum; held-out format = same test items",
    "label_rule": "verified by execution, curated, or an exact edit of a verified example; nothing truncated",
    "encoding": "rows carry enc for target_model at model_revision: Clef (one sequence, question and option spans) or d1 (one sequence per question, option token groups); each with the gold option index",
    "heldout": {"style": CFG["heldout_style"]},
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
rep.to_csv(OUT / "build_report.csv", index=False)
print(json.dumps(manifest["rows"], indent=2))

In [ ]:
assert CHECKS_PASSED, "the hard checks did not pass; fix them before writing or logging the dataset"
if CFG["log_to_wandb"]:
    import wandb
    with wandb.init(project=CFG["wandb_project"], job_type="build-dataset", config=CFG) as run:
        art = wandb.Artifact(CFG["artifact_name"], type="dataset", metadata=manifest)
        art.add_dir(str(OUT))
        run.log_artifact(art)